# Stage 3. Paired tests and robustness diagnostics

Every method predicts the same withheld fires, so all comparisons are paired.
Absolute errors go to Wilcoxon, correlations to Steiger, and each family of
p-values is Holm-adjusted together.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from fireanalogs.config import Config

cfg = Config.load(Path.cwd().parent / "config" / "default.yaml")
DATA = Path.cwd().parent / cfg["paths"]["data_root"]
RESULTS = Path.cwd().parent / cfg["paths"]["results_root"]
RESULTS.mkdir(parents=True, exist_ok=True)
print("data:", DATA)

In [ ]:
import pandas as pd
from fireanalogs import similarity, statistics, robustness
from fireanalogs.library import embedding_matrix, fuel_slope_matrix

library = pd.read_csv(RESULTS / "fire_library_merged.csv", parse_dates=["event_date"])
loo = pd.read_csv(RESULTS / "loo_results.csv", parse_dates=["event_date"])
cosine = similarity.cosine_similarity_matrix(embedding_matrix(library, cfg.n_emb_dims))

## Embedding retrieval against each baseline

In [ ]:
comparison = statistics.compare_methods(loo)
comparison.to_csv(RESULTS / "method_comparison_wilcoxon_steiger_holm.csv", index=False)
comparison

## Sensitivity to analog neighbourhood size

The published K is the smallest value at which error has substantially
stabilised, which shows the result is not driven by a narrow neighbourhood or
by one highly similar fire.

In [ ]:
k_table = robustness.k_sensitivity(library, cosine, cfg.k_list, prior_only=cfg.prior_only)
k_table.to_csv(RESULTS / "k_sensitivity.csv", index=False)
k_table.round(4)

## Fuel by slope transfer

Retrieval is held at embedding similarity, and each withheld fire is predicted
by averaging the 16 fuel by slope probability cells of its K analogs.

In [ ]:
transfer = robustness.fuel_slope_transfer(
    library, cosine, fuel_slope_matrix(library),
    k=cfg.k_default, prior_only=cfg.prior_only,
)
transfer.to_csv(RESULTS / "fuel_slope_transfer.csv", index=False)
robustness.summarise_fuel_slope_transfer(transfer)

## Do ensembles help?

Mixed results here support the reading that the learned representation carries
information the baselines do not, and that averaging it with weaker criteria
dilutes rather than improves it.

In [ ]:
loo_ens = robustness.add_ensembles(loo)
ensemble_table = robustness.compare_ensembles(loo_ens)
ensemble_table.to_csv(RESULTS / "ensemble_comparison_holm.csv", index=False)
ensemble_table